## 1. 준비

In [ ]:
!pip -q install tifffile

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv, glob, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from PIL import Image
import tifffile

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x

print('준비 완료')

## 2. 경로 및 설정

In [ ]:
# ── (1) 데이터셋 ──────────────────────────────────────────────────────────
DATASET_PATH = '/content/drive/MyDrive/AD_dataset'
CATEGORY     = 'Mounting_Plate'

# ── (2) EfficientAD ───────────────────────────────────────────────────────
EFFICIENTAD_MAPS = '/content/drive/MyDrive/efficientAD_application/1_128_256/anomaly_maps/Mounting_Plate/test'

# ── (3) PatchCore ─────────────────────────────────────────────────────────
PATCHCORE_DIR = '/content/drive/MyDrive/branch_analysis_results'

# ── (4) DRAEM ─────────────────────────────────────────────────────────────
DRAEM_DIR = '/content/drive/MyDrive/draem_maps'

# ── (5) 그림에 쓸 이름 (폴더명·제목에 들어감) ─────────────────────────────
NAME_EFF   = 'EfficientAD'
NAME_PC    = 'PatchCore'
NAME_DRAEM = 'DRAEM'

# ── (6) 출력 ──────────────────────────────────────────────────────────────
OUT_DIR = '/content/drive/MyDrive/heatmap_comparison_3method'

# ── (7) 좌표계 (PatchCore summary.json 에서 읽고, 없으면 아래 값) ─────────
FALLBACK_RESIZE_SHORT = 256
FALLBACK_CROP         = 224

# ── (8) 그림 설정 ─────────────────────────────────────────────────────────
SHOW = None          # None 이면 자동 선택. 예: [('good', '003'), ('defect', '004')]
CMAP_STYLE   = 'jet'   # 'diverging' | 'jet'
VMAX_CAP     = 2.0     # 통합 그림 색 축 상한 (판정선의 몇 배까지)
SAVE_PANELS  = True
DRAW_CONTOUR = True
PANEL_DPI    = 200

EXPORT_ALL   = True
EXPORT_LOG   = True
EXPORT_RANGE = (0.25, 32.0)

os.makedirs(OUT_DIR, exist_ok=True)
print('설정 완료 ->', OUT_DIR)

## 3. 맵 폴더 · 공통 이미지 목록

In [ ]:
def _latest(pattern):
    files = sorted(glob.glob(pattern))
    return files[-1] if files else None


def maps_from_summary(base_dir, prefix):
    sp = _latest(os.path.join(base_dir, f'{prefix}_{CATEGORY}_*_summary.json'))
    if sp is None:
        raise FileNotFoundError(
            f'{base_dir} 에 {prefix}_{CATEGORY}_*_summary.json 이 없습니다.')
    with open(sp, encoding='utf-8') as f:
        s = json.load(f)
    md_ = s.get('maps_dir') or sp[:-len('_summary.json')] + '_maps'
    if not os.path.isdir(md_):
        md_ = sp[:-len('_summary.json')] + '_maps'
    return md_, s, sp


PC_MAPS, PC_SUMMARY, _pc_sp = maps_from_summary(PATCHCORE_DIR, 'patchcore')
DR_MAPS, DR_SUMMARY, _dr_sp = maps_from_summary(DRAEM_DIR, 'draem')

METHODS = [
    {'name': NAME_EFF,   'maps_dir': EFFICIENTAD_MAPS, 'summary': None},
    {'name': NAME_PC,    'maps_dir': PC_MAPS,          'summary': PC_SUMMARY},
    {'name': NAME_DRAEM, 'maps_dir': DR_MAPS,          'summary': DR_SUMMARY},
]

for M in METHODS:
    ok = os.path.isdir(M['maps_dir'])
    print(f"  {M['name']:<12} {'OK ' if ok else '없음'} {M['maps_dir']}")
    if not ok:
        raise FileNotFoundError(f"{M['name']} 의 map 폴더가 없습니다.")


def list_keys(maps_dir):
    keys = set()
    for sub in sorted(os.listdir(maps_dir)):
        d = os.path.join(maps_dir, sub)
        if not os.path.isdir(d):
            continue
        for p in glob.glob(os.path.join(d, '*.tif*')):
            keys.add(f'{sub}/' + os.path.splitext(os.path.basename(p))[0])
    return keys


key_sets = {M['name']: list_keys(M['maps_dir']) for M in METHODS}
common = sorted(set.intersection(*key_sets.values()))

print()
for n, ks in key_sets.items():
    extra = sorted(ks - set(common))
    print(f'  {n:<12} {len(ks)}장' + (f'  [이 방법에만: {len(extra)}장 {extra[:3]}]' if extra else ''))
if not common:
    raise ValueError('공통 이미지가 없습니다. CATEGORY 와 경로를 확인하세요.')
print(f'  공통 {len(common)}장')

items = pd.DataFrame({'key': common})
items['subdir'] = items['key'].str.split('/').str[0]
items['file']   = items['key'].str.split('/').str[1]
items['label']  = (items['subdir'] != 'good').astype(int)
print(f"  good {int((items['label'] == 0).sum())}장 / "
      f"defect {int((items['label'] == 1).sum())}장")

## 4. 좌표계 정합

In [ ]:
_frame = (PC_SUMMARY or {}).get('frame', {})
RESIZE_SHORT = _frame.get('resize', FALLBACK_RESIZE_SHORT)
if isinstance(RESIZE_SHORT, (list, tuple)):
    RESIZE_SHORT = RESIZE_SHORT[0]
CROP = int(_frame.get('imagesize', FALLBACK_CROP))
print(f'정합 기준: Resize(짧은 변 {RESIZE_SHORT}) -> CenterCrop({CROP})')


def _resized_hw(h, w, size):
    if w <= h:
        return int(size * h / w), size
    return size, int(size * w / h)


def _crop_box(h, w, crop):
    return int(round((h - crop) / 2.0)), int(round((w - crop) / 2.0))


def to_crop_frame_image(pil_img):
    w, h = pil_img.size
    nh, nw = _resized_hw(h, w, RESIZE_SHORT)
    img = pil_img.resize((nw, nh), Image.BILINEAR)
    top, left = _crop_box(nh, nw, CROP)
    return img.crop((left, top, left + CROP, top + CROP))


def to_crop_frame_array(a, nearest=False):
    h, w = a.shape[:2]
    if (h, w) == (CROP, CROP):
        return np.asarray(a, dtype=np.float64)
    nh, nw = _resized_hw(h, w, RESIZE_SHORT)
    im = Image.fromarray(np.asarray(a, dtype=np.float32), mode='F')
    im = im.resize((nw, nh), Image.NEAREST if nearest else Image.BILINEAR)
    top, left = _crop_box(nh, nw, CROP)
    return np.asarray(im.crop((left, top, left + CROP, top + CROP)),
                      dtype=np.float64)


def find_original(subdir, name):
    for ext in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'):
        p = os.path.join(DATASET_PATH, CATEGORY, 'test', subdir, name + ext)
        if os.path.exists(p):
            return p
    hits = glob.glob(os.path.join(DATASET_PATH, CATEGORY, 'test', subdir, name + '.*'))
    return hits[0] if hits else None


def find_gt(subdir, name):
    gt = os.path.join(DATASET_PATH, CATEGORY, 'ground_truth')
    for cand in (os.path.join(gt, subdir, f'{name}_mask.png'),
                 os.path.join(gt, f'{name}_mask.png')):
        if os.path.exists(cand):
            return cand
    return None


def load_raw_map(M, subdir, name):
    for ext in ('.tiff', '.tif'):
        p = os.path.join(M['maps_dir'], subdir, name + ext)
        if os.path.exists(p):
            m = tifffile.imread(p).astype(np.float64)
            if m.ndim > 2:
                m = m.squeeze()
            return to_crop_frame_array(m)
    raise FileNotFoundError(f"{M['maps_dir']}/{subdir}/{name}.tiff 가 없습니다")


def load_gt(subdir, name, label):
    if label == 0:
        return np.zeros((CROP, CROP), dtype=bool)
    p = find_gt(subdir, name)
    if p is None:
        return None
    g = np.asarray(Image.open(p).convert('L'), dtype=np.float64)
    return to_crop_frame_array(g, nearest=False) > 127.5


_probe = items.iloc[0]
_op = find_original(_probe['subdir'], _probe['file'])
if _op:
    _w, _h = Image.open(_op).size
    print(f'원본 이미지 크기: {_w}x{_h}', end='  ')
    if _w != _h:
        print('\n  [경고] 정사각형이 아닙니다. DRAEM 은 종횡비를 무시하고 리사이즈하므로 '
              '다른 두 방법과 같은 영역을 보지 않습니다.')
    else:
        print('(정사각형 -> DRAEM 좌표계 전제 충족)')

print()
for M in METHODS:
    _m = load_raw_map(M, _probe['subdir'], _probe['file'])
    print(f"  {M['name']:<12} map shape {_m.shape}   raw range "
          f"[{_m.min():.6f}, {_m.max():.6f}]")
print('세 shape 이 모두 같아야 합니다.')

## 5. 픽셀 판정선 계산

In [ ]:
def compute_f1_max_threshold(normal_scores, anomaly_scores):
    normal_sorted = np.sort(normal_scores)
    anomaly_sorted = np.sort(anomaly_scores)
    thresholds = np.unique(anomaly_sorted)

    n_norm, n_anom = len(normal_sorted), len(anomaly_sorted)
    tp = n_anom - np.searchsorted(anomaly_sorted, thresholds, side='left')
    fp = n_norm - np.searchsorted(normal_sorted, thresholds, side='left')
    f1 = 2.0 * tp / (tp + fp + n_anom)

    best = int(np.argmax(f1))
    return float(thresholds[best]), float(f1[best])


print('픽셀 점수 수집 중 (공통 crop 좌표계)...')
gt_cache, raw_cache = {}, {}
missing_gt = []

for _, it in tqdm(list(items.iterrows()), total=len(items), desc='loading'):
    key = it['key']
    g = load_gt(it['subdir'], it['file'], int(it['label']))
    if g is None:
        missing_gt.append(key)
        continue
    gt_cache[key] = g
    for M in METHODS:
        raw_cache[(M['name'], key)] = load_raw_map(M, it['subdir'], it['file'])

if missing_gt:
    print(f'  [주의] GT 마스크를 찾지 못해 제외한 {len(missing_gt)}장: {missing_gt[:5]}')

keys_ok = [k for k in items['key'] if k in gt_cache]
if not keys_ok:
    raise ValueError('GT 를 찾은 이미지가 없습니다. ground_truth 경로를 확인하세요.')

THRESH, PIXSTATS = {}, {}
for M in METHODS:
    n = M['name']
    normal = np.concatenate([raw_cache[(n, k)][~gt_cache[k]].ravel() for k in keys_ok])
    anom   = np.concatenate([raw_cache[(n, k)][gt_cache[k]].ravel() for k in keys_ok
                             if gt_cache[k].any()])
    thr, f1 = compute_f1_max_threshold(normal, anom)
    if thr <= 0:
        raise RuntimeError(f'{n}: 픽셀 판정선이 {thr} 로 양수가 아닙니다.')

    fp = int(np.count_nonzero(normal >= thr))
    tp = int(np.count_nonzero(anom >= thr))
    M['threshold'] = thr
    THRESH[n] = thr
    PIXSTATS[n] = {
        'pixel_f1_max': f1, 'threshold': thr,
        'n_normal': len(normal), 'n_anomaly': len(anom),
        'tp': tp, 'fp': fp, 'fn': len(anom) - tp,
        'precision': tp / (tp + fp) if tp + fp else 0.0,
        'recall': tp / len(anom) if len(anom) else 0.0,
        'fpr': fp / len(normal) if len(normal) else 0.0,
        'pred_gt_area_ratio': (tp + fp) / len(anom) if len(anom) else float('nan'),
    }

print()
print(f"{'method':<14}{'threshold':>12}{'pixel F1':>10}{'prec':>8}{'recall':>8}"
      f"{'FPR':>10}{'Pred/GT area':>14}")
for M in METHODS:
    s = PIXSTATS[M['name']]
    print(f"{M['name']:<14}{s['threshold']:>12.6f}{s['pixel_f1_max']:>10.4f}"
          f"{s['precision']:>8.3f}{s['recall']:>8.3f}{s['fpr']:>10.5f}"
          f"{s['pred_gt_area_ratio']:>13.2f}x")
print()
print('[참고] threshold 절대값은 비교하지 마세요. 축이 다릅니다.')
print('       비교가 성립하는 것은 map/threshold 로 나눈 뒤의 그림과 비율뿐입니다.')

## 6. 통합 그림

In [ ]:
def norm_map(name, key):
    return raw_cache[(name, key)] / THRESH[name]


rows = []
for k in keys_ok:
    sub, nm = k.split('/', 1)
    gt_px = int(gt_cache[k].sum())
    r = {'key': k, 'subdir': sub, 'file': nm,
         'label': int(items.set_index('key').loc[k, 'label']), 'gt_px': gt_px}
    for M in METHODS:
        n = M['name']
        m = norm_map(n, k)
        pred_px = int(np.count_nonzero(m >= 1.0))
        r[f'peak_{n}'] = float(m.max())
        r[f'predpx_{n}'] = pred_px
        r[f'arearatio_{n}'] = (pred_px / gt_px) if gt_px else np.nan
    rows.append(r)

per_image = pd.DataFrame(rows).set_index('key')

if SHOW is None:
    pcols = [f'predpx_{M["name"]}' for M in METHODS]
    spread = per_image[pcols].max(axis=1) - per_image[pcols].min(axis=1)
    SHOW_KEYS = list(spread.nlargest(4).index)
    print('자동 선택 (예측 면적 편차 상위 4장):')
else:
    SHOW_KEYS = [f'{s}/{n}' for s, n in SHOW]
    missing = [k for k in SHOW_KEYS if k not in per_image.index]
    if missing:
        raise ValueError(f'목록에 없는 항목: {missing}')
    print('직접 지정:')

for k in SHOW_KEYS:
    print(f"  {k:<24} GT {int(per_image.loc[k, 'gt_px']):>6}px  |  " +
          '  '.join(f"{M['name']} {int(per_image.loc[k, 'predpx_' + M['name']]):>6}px"
                    for M in METHODS))

In [ ]:
SURFACE, INK, MUTED = '#fcfcfb', '#1a1a19', '#6b6b66'

if CMAP_STYLE == 'jet':
    CMAP = plt.get_cmap('jet')
elif CMAP_STYLE == 'diverging':
    CMAP = LinearSegmentedColormap.from_list(
        'safe_alarm', ['#0d3c6e', '#2a78d6', '#c9c9c4', '#eb6834', '#8f2f10'])
else:
    raise ValueError(f"CMAP_STYLE 은 'diverging' 또는 'jet' 이어야 합니다: {CMAP_STYLE}")

CONTOUR_COLOR = '#ffffff' if CMAP_STYLE == 'jet' else INK

vmax = VMAX_CAP
NORM = TwoSlopeNorm(vmin=0.0, vcenter=1.0, vmax=vmax)
CB_TICKS = [0, 0.5, 1.0, vmax]
CB_LABELS = ['0', '0.5', '1.0 (pixel threshold)', f'{vmax:.1f}']

EPS = 1e-9
_lo, _hi = float(np.log2(EXPORT_RANGE[0])), float(np.log2(EXPORT_RANGE[1]))
if EXPORT_LOG:
    NORM_EXPORT = TwoSlopeNorm(vmin=_lo, vcenter=0.0, vmax=_hi)
    EXPORT_TICKS = [t for t in range(int(np.floor(_lo)), int(np.ceil(_hi)) + 1)
                    if _lo <= t <= _hi]
    EXPORT_LABELS = [f'{2.0 ** t:g}x' + (' (threshold)' if t == 0 else '')
                     for t in EXPORT_TICKS]
    EXPORT_CONTOUR = 0.0
else:
    NORM_EXPORT, EXPORT_TICKS, EXPORT_LABELS = NORM, CB_TICKS, CB_LABELS
    EXPORT_CONTOUR = 1.0


def to_display(m):
    return np.log2(np.clip(m, EPS, None)) if EXPORT_LOG else m


print(f'색 축: 통합 그림 선형 0~{vmax:.1f}, '
      f'일괄 저장 {"log2 " + str(EXPORT_RANGE) if EXPORT_LOG else "선형 동일"}')
print()
print('방법별 ratio 상한 (전체 이미지의 최댓값):')
for M in METHODS:
    n = M['name']
    print(f"  {n:<12} {per_image[f'peak_{n}'].max():.2f}x")
print('  값이 크게 다르면 선형 축에서 낮은 쪽이 불리하게 보입니다. log2 축을 권합니다.')

In [ ]:
def title_for(M, key):
    n = M['name']
    r = per_image.loc[key]
    t = f"{n}   peak {r[f'peak_{n}']:.2f}x   {int(r[f'predpx_{n}'])}px"
    if r['gt_px'] > 0:
        t += f"   ({r[f'arearatio_{n}']:.2f}x GT)"
    return t


ncol = 1 + len(METHODS)
fig, axes = plt.subplots(len(SHOW_KEYS), ncol,
                         figsize=(3.4 * ncol, 3.4 * len(SHOW_KEYS)),
                         dpi=140, squeeze=False)
fig.patch.set_facecolor(SURFACE)
fig.subplots_adjust(top=0.95, hspace=0.12, wspace=0.04)

im = None
for r, key in enumerate(SHOW_KEYS):
    subdir, name = key.split('/', 1)

    ax = axes[r][0]
    op = find_original(subdir, name)
    if op:
        ax.imshow(np.array(to_crop_frame_image(Image.open(op).convert('RGB'))))
    g = gt_cache[key]
    if g.any():
        ax.contour(g.astype(float), levels=[0.5], colors=['#00ff88'],
                   linewidths=1.2)
    lab = 'defect' if int(per_image.loc[key, 'label']) == 1 else 'good'
    ax.set_title(f'{key}  ({lab})   GT {int(per_image.loc[key, "gt_px"])}px',
                 color=INK, fontsize=10, loc='left')
    ax.axis('off')

    for c, M in enumerate(METHODS, start=1):
        ax = axes[r][c]
        m = norm_map(M['name'], key)
        im = ax.imshow(m, cmap=CMAP, norm=NORM)
        if m.max() > 1.0 > m.min():
            ax.contour(m, levels=[1.0], colors=[CONTOUR_COLOR], linewidths=1.0)
        ax.set_title(title_for(M, key), color=INK, fontsize=10, loc='left')
        ax.axis('off')

cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, ticks=CB_TICKS)
cbar.ax.set_yticklabels(CB_LABELS, fontsize=9, color=MUTED)
cbar.outline.set_visible(False)

fig.suptitle('Anomaly maps normalized by each method’s own PIXEL decision line '
             '(1.0 = pixel F1-max threshold, CenterCrop frame; green = GT)',
             color=INK, fontsize=12, x=0.02, y=0.985, ha='left', va='top')

out = os.path.join(OUT_DIR, f'pixelnorm_heatmaps_{CATEGORY}_{CMAP_STYLE}.png')
fig.savefig(out, facecolor=SURFACE, bbox_inches='tight')
plt.show()
print('저장:', out)

## 7. 낱장 저장 / 전체 일괄 저장

In [ ]:
def save_bare(arr, path, as_map, norm=None, contour_at=1.0, gt=None):
    norm = NORM if norm is None else norm
    h, w = arr.shape[:2]
    f = plt.figure(figsize=(w / PANEL_DPI, h / PANEL_DPI), dpi=PANEL_DPI)
    ax_ = f.add_axes([0, 0, 1, 1])
    ax_.set_axis_off()
    if as_map:
        ax_.imshow(arr, cmap=CMAP, norm=norm)
        if DRAW_CONTOUR and arr.max() > contour_at > arr.min():
            ax_.contour(arr, levels=[contour_at], colors=[CONTOUR_COLOR],
                        linewidths=1.0)
    else:
        ax_.imshow(arr)
    if gt is not None and gt.any():
        ax_.contour(gt.astype(float), levels=[0.5], colors=['#00ff88'],
                    linewidths=1.0)
    f.savefig(path, dpi=PANEL_DPI, facecolor=SURFACE)
    plt.close(f)


def save_colorbar(path, norm=None, ticks=None, labels=None):
    from matplotlib.cm import ScalarMappable
    norm = NORM if norm is None else norm
    ticks = CB_TICKS if ticks is None else ticks
    labels = CB_LABELS if labels is None else labels
    f = plt.figure(figsize=(1.9, 4.6), dpi=PANEL_DPI)
    f.patch.set_facecolor(SURFACE)
    cax = f.add_axes([0.06, 0.05, 0.24, 0.90])
    cb = f.colorbar(ScalarMappable(norm=norm, cmap=CMAP), cax=cax, ticks=ticks)
    cb.ax.set_yticklabels(labels, fontsize=9, color=MUTED)
    cb.outline.set_visible(False)
    f.savefig(path, facecolor=SURFACE, bbox_inches='tight')
    plt.close(f)


if SAVE_PANELS:
    panel_dir = os.path.join(OUT_DIR, f'panels_{CATEGORY}_{CMAP_STYLE}')
    os.makedirs(panel_dir, exist_ok=True)

    index_rows = []
    for key in SHOW_KEYS:
        subdir, name = key.split('/', 1)
        op = find_original(subdir, name)
        if op:
            p = os.path.join(panel_dir, f'{subdir}_{name}_original.png')
            save_bare(np.array(to_crop_frame_image(Image.open(op).convert('RGB'))),
                      p, as_map=False, gt=gt_cache[key])
            index_rows.append({'subdir': subdir, 'file': name, 'panel': 'original',
                               'peak': '', 'pred_px': int(per_image.loc[key, 'gt_px']),
                               'area_ratio_vs_gt': '', 'png': os.path.basename(p)})

        for M in METHODS:
            n = M['name']
            m = norm_map(n, key)
            p = os.path.join(panel_dir, f'{subdir}_{name}_{n}.png')
            save_bare(m, p, as_map=True)
            ar = per_image.loc[key, f'arearatio_{n}']
            index_rows.append({
                'subdir': subdir, 'file': name, 'panel': n,
                'peak': round(float(per_image.loc[key, f'peak_{n}']), 4),
                'pred_px': int(per_image.loc[key, f'predpx_{n}']),
                'area_ratio_vs_gt': ('' if np.isnan(ar) else round(float(ar), 4)),
                'png': os.path.basename(p)})

    save_colorbar(os.path.join(panel_dir, 'colorbar.png'))

    with open(os.path.join(panel_dir, 'panels_index.csv'), 'w', newline='',
              encoding='utf-8-sig') as f:
        w_ = csv.DictWriter(f, fieldnames=['subdir', 'file', 'panel', 'peak',
                                           'pred_px', 'area_ratio_vs_gt', 'png'])
        w_.writeheader()
        w_.writerows(index_rows)

    print(f'낱장 {len(index_rows)}장 + 컬러바 1장 저장 -> {panel_dir}')
    print('  original 패널의 pred_px 열에는 GT 면적이 들어갑니다')

In [ ]:
if EXPORT_ALL:
    root = os.path.join(OUT_DIR, f'heat_map_{CATEGORY}_{CMAP_STYLE}'
                                 + ('_log' if EXPORT_LOG else ''))
    subs = sorted(items['subdir'].unique())
    for folder in [M['name'] for M in METHODS] + ['original']:
        for sub in subs:
            os.makedirs(os.path.join(root, folder, sub), exist_ok=True)

    print(f'전체 {len(keys_ok)}장 x {len(METHODS)}방법 저장 중 -> {root}')
    if EXPORT_LOG:
        print(f'  색 축: log2  {EXPORT_RANGE[0]:g}x ~ {EXPORT_RANGE[1]:g}x '
              f'(중앙 1.0x = 픽셀 판정선, 한 칸 = 2배)')

    all_rows = []
    for key in tqdm(keys_ok, desc='exporting'):
        subdir, name = key.split('/', 1)
        r = per_image.loc[key]

        op = find_original(subdir, name)
        if op:
            save_bare(np.array(to_crop_frame_image(Image.open(op).convert('RGB'))),
                      os.path.join(root, 'original', subdir, f'{name}.png'),
                      as_map=False, gt=gt_cache[key])

        row = {'subdir': subdir, 'file': name, 'label': int(r['label']),
               'gt_px': int(r['gt_px'])}
        for M in METHODS:
            n = M['name']
            m = norm_map(n, key)
            save_bare(to_display(m),
                      os.path.join(root, n, subdir, f'{name}.png'),
                      as_map=True, norm=NORM_EXPORT, contour_at=EXPORT_CONTOUR)
            ar = r[f'arearatio_{n}']
            row[f'peak_{n}'] = round(float(r[f'peak_{n}']), 4)
            row[f'predpx_{n}'] = int(r[f'predpx_{n}'])
            row[f'arearatio_{n}'] = ('' if np.isnan(ar) else round(float(ar), 4))
            row[f'clipped_{n}'] = 'Y' if r[f'peak_{n}'] > EXPORT_RANGE[1] else ''
        all_rows.append(row)

    save_colorbar(os.path.join(root, 'colorbar.png'),
                  norm=NORM_EXPORT, ticks=EXPORT_TICKS, labels=EXPORT_LABELS)

    names = [M['name'] for M in METHODS]
    fields = ['subdir', 'file', 'label', 'gt_px'] + [
        f'{k}_{n}' for n in names
        for k in ('peak', 'predpx', 'arearatio', 'clipped')]
    with open(os.path.join(root, 'index.csv'), 'w', newline='',
              encoding='utf-8-sig') as f:
        w_ = csv.DictWriter(f, fieldnames=fields, restval='')
        w_.writeheader()
        w_.writerows(all_rows)

    with open(os.path.join(root, 'pixel_thresholds.json'), 'w',
              encoding='utf-8') as f:
        json.dump({'category': CATEGORY,
                   'frame': {'resize_short': RESIZE_SHORT, 'crop': CROP},
                   'normalization': 'map / pixel_f1_max_threshold '
                                    '(1.0 = pixel decision boundary)',
                   'per_method': PIXSTATS}, f, indent=2, ensure_ascii=False)

    print(f'  히트맵 {len(all_rows) * len(names)}장 + 원본 {len(all_rows)}장 + 컬러바 1장')
    print('  index.csv / pixel_thresholds.json 저장 완료')

    print()
    _idx = pd.DataFrame(all_rows)
    _def = _idx[_idx['label'] == 1]
    print('결함 이미지 기준 Pred/GT 면적 배율 (1 보다 크면 과분할):')
    for n in names:
        v = pd.to_numeric(_def[f'arearatio_{n}'], errors='coerce').dropna()
        if len(v):
            print(f'  {n:<12} 중앙값 {v.median():5.2f}x   평균 {v.mean():5.2f}x   '
                  f'최대 {v.max():5.2f}x')